# BigBasket Capstone — Python/Pandas Cleaning & Analysis
This notebook independently cleans the messy raw export and cross-checks the SQL findings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
orders = pd.read_csv('orders_raw.csv')
products = pd.read_csv('products.csv')
orders.info()
orders.describe(include='all')
orders['status'].value_counts(dropna=False)

## Initial observations
The raw export contains duplicate order IDs, inconsistent casing/whitespace, missing amounts/ratings, and suspiciously large amount values.

In [ ]:
before=len(orders)
orders=orders.drop_duplicates(subset='order_id',keep='first').copy()
print('Rows before:',before,'Rows after:',len(orders),'Removed:',before-len(orders))
orders['city']=orders['city'].astype('string').str.strip().str.title()
orders['category']=orders['category'].astype('string').str.strip().str.title()
print(sorted(orders['city'].dropna().unique()))
print(sorted(orders['category'].dropna().unique()))

In [ ]:
orders['amount_inr']=pd.to_numeric(orders['amount_inr'],errors='coerce')
orders['order_date']=pd.to_datetime(orders['order_date'],errors='coerce')
# Revenue logic: use delivered orders with non-null amounts; cap delivered amounts using IQR.
delivered=orders[orders['status'].eq('Delivered')].copy()
q1=delivered['amount_inr'].quantile(.25); q3=delivered['amount_inr'].quantile(.75); iqr=q3-q1
upper=q3+1.5*iqr
delivered['amount_capped']=delivered['amount_inr'].clip(upper=upper)
analysis=delivered.dropna(subset=['amount_capped']).merge(products,on='product_id',how='left')
analysis['month']=analysis['order_date'].dt.strftime('%Y-%m')
summary=analysis.groupby('category_x',as_index=False).agg(total_revenue=('amount_capped','sum'),order_count=('order_id','count'))
summary.sort_values('total_revenue',ascending=False)

In [ ]:
monthly=analysis.groupby(['month'],as_index=False)['amount_capped'].sum()
plt.figure(figsize=(9,4)); plt.plot(monthly['month'],monthly['amount_capped'],marker='o'); plt.xticks(rotation=45); plt.title('Monthly Delivered Revenue'); plt.xlabel('Month'); plt.ylabel('Revenue (INR)'); plt.tight_layout(); plt.show()

In [ ]:
cat=analysis.groupby('category_x',as_index=False)['amount_capped'].sum().sort_values('amount_capped',ascending=False)
plt.figure(figsize=(9,4)); plt.bar(cat['category_x'],cat['amount_capped']); plt.xticks(rotation=45,ha='right'); plt.title('Revenue by Category'); plt.ylabel('Revenue (INR)'); plt.tight_layout(); plt.show()

In [ ]:
city=analysis.groupby('city',as_index=False)['amount_capped'].sum().sort_values('amount_capped',ascending=False)
plt.figure(figsize=(7,4)); plt.bar(city['city'],city['amount_capped']); plt.title('Revenue by City'); plt.ylabel('Revenue (INR)'); plt.tight_layout(); plt.show()

## Three insights
1. **What:** Household Essentials is the highest-revenue category. **Why:** It has the largest category total in the SQL report. **Next step:** protect stock availability and negotiate supplier volume discounts.
2. **What:** Personal Care is the second-highest category. **Why:** Its revenue is materially above the remaining categories. **Next step:** test bundles and targeted repeat-purchase offers.
3. **What:** Lower-revenue categories need attention. **Why:** Fruits & Vegetables and Snacks & Beverages trail the leaders. **Next step:** review assortment, pricing, and promotion performance by month.